# Klike — Recommendations Engine interativo

Esta é a interface de demonstração da Parte 3. Execute as células no VS Code ou Jupyter com o ambiente de `requirements.txt`, preencha os campos e clique em **Gerar recomendações**. O GitHub não executa widgets em sua visualização estática.

O modelo Ridge da Parte 2 é reutilizado sem retreinamento, e as evidências são calculadas somente com as 400 campanhas de treino. As estimativas não representam efeitos causais.


## 1. Carregar modelo e base de referência


In [ ]:
from pathlib import Path
import sys
import html
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

PROJECT_ROOT = next(
    (p for p in (Path.cwd(), Path.cwd().parent)
     if (p / "src" / "recommendations_engine.py").is_file()), None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Abra a partir da raiz do projeto ou da pasta notebooks/.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.recommendations_engine import RecommendationsEngine

engine = RecommendationsEngine(PROJECT_ROOT)
print("Ridge reduzido carregado.")
print("Campanhas de treinamento para evidência contextual:", len(engine.df_evidencias))


## 2. Informar nova campanha


In [ ]:
def dropdown(label, options, value):
    return widgets.Dropdown(
        description=label, options=options, value=value,
        style={"description_width": "130px"},
        layout=widgets.Layout(width="350px")
    )

identificador = widgets.Text(value="DEMO-MANUAL-001", description="Campanha")
plataforma = dropdown("Plataforma", engine.categorias_validas["platform"], "Meta")
categoria = dropdown("Categoria", engine.categorias_validas["category"], "E-commerce")
formato = dropdown("Formato", engine.categorias_validas["format"], "vertical")
densidade = dropdown("Densidade texto", engine.categorias_validas["text_density"], "low")
duracao = widgets.BoundedIntText(value=16, min=1, max=1000, description="Duração (s)")
retargeting = widgets.Checkbox(description="Retargeting", value=False, indent=False)
hook = widgets.Checkbox(description="Hook nos 3 segundos", value=False, indent=False)
rosto = widgets.Checkbox(description="Rosto humano", value=False, indent=False)
legendas = widgets.Checkbox(description="Legendas", value=False, indent=False)
cta = widgets.Checkbox(description="Call-to-action (CTA)", value=False, indent=False)
botao = widgets.Button(description="Gerar recomendações", button_style="success", icon="check")
saida = widgets.Output(layout=widgets.Layout(border="1px solid #dddddd", padding="12px"))

def ler_campanha():
    return {
        "campaign_id": identificador.value.strip() or "Campanha sem identificador",
        "platform": plataforma.value,
        "category": categoria.value,
        "is_retargeting": retargeting.value,
        "video_duration_s": duracao.value,
        "format": formato.value,
        "has_subtitle": legendas.value,
        "has_cta": cta.value,
        "has_hook": hook.value,
        "has_face": rosto.value,
        "text_density": densidade.value,
    }

def gerar(_):
    botao.disabled = True
    with saida:
        clear_output(wait=True)
        try:
            resultado = engine.recomendar_campanha(
                ler_campanha(), max_recomendacoes=3,
                delta_minimo=1.0, n_bootstrap=2000
            )
            nome = html.escape(str(resultado["campaign_id"]))
            display(HTML(
                f"<h3>{nome}</h3>"
                f"<p>Score previsto: <b>{resultado['score_original']:.2f}</b> | "
                f"Cenários: {resultado['total_cenarios']} | "
                f"Recomendações elegíveis: {resultado['total_recomendacoes']}</p>"
            ))
            for rec in resultado["recomendacoes"]:
                texto = html.escape(rec["texto"])
                display(HTML(
                    "<div style='padding:12px;margin:12px 0;"
                    "border-left:4px solid #16835c;background:#f7faf8;color:#20272c'>"
                    f"<b>{rec['prioridade']}ª recomendação</b><p>{texto}</p>"
                    f"<small>Amostra: N={rec['n_original']} vs N={rec['n_proposto']}</small></div>"
                ))
            if resultado["total_recomendacoes"] < 3:
                display(HTML(
                    "<p><em>Menos de três melhorias elegíveis. "
                    "Não foram inventados ganhos para completar a lista.</em></p>"
                ))
            display(HTML(
                "<p><small>Diferenças entre previsões e associações observacionais "
                "não são ganhos causais comprovados. O bootstrap não controla "
                "confundimento ou múltiplos testes.</small></p>"
            ))
        except (ValueError, TypeError, KeyError) as erro:
            print(f"Entrada inválida: {erro}")
        finally:
            botao.disabled = False

botao.on_click(gerar)
display(widgets.VBox([
    widgets.HBox([
        widgets.VBox([identificador, plataforma, categoria, formato, densidade, duracao]),
        widgets.VBox([retargeting, hook, rosto, legendas, cta])
    ], layout=widgets.Layout(flex_flow="row wrap", gap="20px")),
    botao, saida
]))


## Interpretação

O delta do Ridge compara duas previsões para versões alternativas de um criativo. As diferenças entre médias dos grupos de treino são observacionais e podem refletir confundimento. As sugestões são avaliadas isoladamente e não devem ser somadas. A função pode retornar menos de três alterações quando faltarem evidências suficientes.

A análise completa está em [parte-3.ipynb](parte-3.ipynb), e o código reutilizável está em [src/recommendations_engine.py](../src/recommendations_engine.py).
